# Search with SyftHub, cell by cell

The Python SDK against a real Hub: browse the catalogue, compose a search over several publishers,
see what each one will cost and need **before** anything is sent, resolve what blocks, run it, and
read the results. Every query goes from this process straight to the publisher's Space; the Hub
only serves the catalogue, the per-Space tokens and, for metered sources, the payment credential.

You need a personal access token from your SyftHub profile. Export it as `SYFTHUB_TOKEN` before
starting the kernel, or the first cell asks for it. `SYFTHUB_URL` points the SDK at another Hub;
without it, `https://syfthub.openmined.org`.

The sync twin `syfthub.Hub` is used throughout because notebooks already run an event loop;
`syfthub.AsyncHub` is the same API with `await`. Paid sources charge real money: the cells below
show every cost before sending, and nothing is paid without the `PAY` switch in section 4.

In [1]:
import os
from getpass import getpass

import syfthub

token = os.environ.get("SYFTHUB_TOKEN") or getpass("SyftHub token: ")
hub = syfthub.Hub(token=token)  # or syfthub.Hub("https://your-hub", token=...)
me = hub.auth.whoami()
print(hub, "|", me.username, me.email)

AsyncHub('https://syfthub.openmined.org', as=maguser) | maguser maguser@openmined.org


## 1. Browse the catalogue

A source is a published endpoint, addressed as `owner/slug`. `hub.sources.list()` reads every page
of the Hub's public listing; `filter`, `matching` and `pick` narrow a listing without another
request. The Hub lists only sources whose Space is reporting in, so what you see here is live.

In [2]:
listing = hub.sources.list()
print(len(listing), "sources\n")
for s in listing:
    price = (
        f"{s.pricing.price} per {s.pricing.unit.value} via {s.pricing.rail.value}"
        if s.pricing
        else "free"
    )
    reach = "" if s.reachable else "  (no Space connected)"
    print(f"{s.path:45} {s.type.value:12} {price}{reach}")

27 sources

putricintobuliah/news-new                     data_source  100 IDR per request via cluster
tempo/tempo-internal                          data_source  free
berita-jatengku/berita-jateng                 data_source  free
putricintobuliah/newstestingg                 data_source  1 IDR per request via cluster
sekretariat/radar-sukabumi                    data_source  free
sekretariat/beritajateng-tv                   data_source  free
sekretariat/berita-jaten-tv                   data_source  free
cantonal-social-services/cantonal-social-services-fsio-public model_data_source free
cantonal-social-services/cantonal-social-services-fsio-casework model_data_source free
cantonal-migration-office/state-secretariat-migration-public model_data_source free
cantonal-migration-office/state-secretariat-migration-caseworker model_data_source free
federal-office-public-health/federal-office-health-public model_data_source free
federal-office-public-health/federal-office-health-casework mod

In [3]:
free = listing.filter(free=True)
print("free:", free.paths)
print("models:", listing.filter(type="model").paths)
print("matching 'news':", listing.matching("news").paths)

free: ('tempo/tempo-internal', 'berita-jatengku/berita-jateng', 'sekretariat/radar-sukabumi', 'sekretariat/beritajateng-tv', 'sekretariat/berita-jaten-tv', 'cantonal-social-services/cantonal-social-services-fsio-public', 'cantonal-social-services/cantonal-social-services-fsio-casework', 'cantonal-migration-office/state-secretariat-migration-public', 'cantonal-migration-office/state-secretariat-migration-caseworker', 'federal-office-public-health/federal-office-health-public', 'federal-office-public-health/federal-office-health-casework', 'cantonax_tax_office/federal-tax-administration-public', 'cantonax_tax_office/federal-tax-administration-caseworker', 'public-ai-vault/publicai-user-vault', 'tempo/tempoassistant-articles-free', 'openmined-models/qwen-25-72b', 'openmined-models/gpt-4o', 'openmined-models/deepseek')
models: ('openmined-models/qwen-25-72b', 'openmined-models/gpt-4o', 'openmined-models/deepseek')
matching 'news': ('tempo/news', 'putricintobuliah/news-new', 'putricintobuli

In [4]:
owner = listing[0].owner_username
hub.sources.list(owner=owner).paths  # the Hub's by-owner route

('putricintobuliah/news-new', 'putricintobuliah/newstestingg')

In [5]:
source = listing[0]
source = hub.sources.get(source.path)  # one source by path, read from the Hub now
print(source.path, "|", source.description)
print("pricing:", source.pricing)
print("access for you:", source.access(me.email))

putricintobuliah/news-new | xxxx
pricing: rail=<Rail.CLUSTER: 'cluster'> prices=(Price(amount=Money(amount=Decimal('100.0'), currency='IDR'), unit=<PriceUnit.REQUEST: 'request'>, policy_type='cluster'),) currency='IDR' applied_to=('*',) wallet_id='6f56beb3-fdf2-4735-b2fb-152dc7e3a659' wallet_owner='sekretariat' bundles=(Bundle(id='Starter', amount=Money(amount=Decimal('10000'), currency='IDR'), credits=None), Bundle(id='Basic', amount=Money(amount=Decimal('50000'), currency='IDR'), credits=None), Bundle(id='Pro', amount=Money(amount=Decimal('100000'), currency='IDR'), credits=None), Bundle(id='Enterprise', amount=Money(amount=Decimal('500000'), currency='IDR'), credits=None)) payment_url='https://station.amsinews.id/api/v1/credits/6f56beb3-fdf2-4735-b2fb-152dc7e3a659/invoices' credits_url='https://station.amsinews.id/api/v1/credits/6f56beb3-fdf2-4735-b2fb-152dc7e3a659/balance' invoices_url='https://station.amsinews.id/api/v1/credits/6f56beb3-fdf2-4735-b2fb-152dc7e3a659/invoices/me'
acc

## 2. Compose a plan

`hub.search` returns a `SearchPlan`. Nothing is sent yet: the plan is a frozen value you can
inspect, save, and change with `skip`. Name the sources you want, or leave them out and the plan
covers every free document source your account may use; paid sources are only searched when
named. `max_sources` caps that automatic pick and never a list you pass.

Set `SOURCES` to your own paths to search paid or private sources; the rest of the notebook works
either way.

In [6]:
QUERY = "What are the main findings?"
SOURCES = None  # e.g. ["owner/slug", "other/slug"]; None lets the Hub pick free sources

plan = hub.search(QUERY, SOURCES, limit=3, similarity_threshold=0.4)
print("explicit sources:", plan.explicit)
plan

explicit sources: False


SearchPlan(query='What are the main findings?', sources=(), limit=3, similarity_threshold=0.4, include_metadata=True, skips=(), max_sources=10, version='2.0.0a1', created_at=datetime.datetime(2026, 10, 13, 10, 7, 27, 605641, tzinfo=datetime.timezone.utc))

## 3. Pre-flight

One row per source with a verdict, and one action per *cause*. `pre.ready` is `True` only when
every row will be sent. A prepaid source whose wallet cannot cover the estimate is held with a
`TopUp`; a source whose access policy excludes you with `AccessDenied`; one over your budget cap
with `OverBudget`; one whose host is cooling down after failures with `CircuitOpen`. Each action
carries the steps that resolve it.

In [7]:
pre = plan.preflight()
for row in pre:
    print(f"{row.path:45} {row.verdict.value:16} {row.estimate!s:14} {row.note or ''}")
print("\nready:", pre.ready, "| estimate for the sending rows:", pre.estimate)

federal-office-public-health/federal-office-health-public ready            0 USD          
cantonal-social-services/cantonal-social-services-fsio-public ready            0 USD          
cantonax_tax_office/federal-tax-administration-public ready            0 USD          
cantonal-migration-office/state-secretariat-migration-public ready            0 USD          

ready: True | estimate for the sending rows: {}


In [8]:
for action in pre.actions:
    print(action.kind.value, "affects", action.paths)
    for step in action.steps:
        print("  -", step)

## 4. Resolve what blocks

Three verbs cover every decision: `plan.skip(...)`, `hub.set_budget(...)`, and for a `TopUp`,
`action.top_up()`. A top-up returns an `Invoice` with a checkout URL; nothing is charged until you
pay it there, and `invoice.wait_paid()` blocks until the Space sees the payment.

The cell below skips every held source so the plan can run. Flip `PAY` to `True` to top up
instead: it opens an invoice on the wallet named by the first `TopUp` and waits for you to pay it.

In [9]:
PAY = False

topups = [a for a in pre.actions if isinstance(a, syfthub.TopUp)]
if PAY and topups:
    topup = topups[0]
    print(
        "wallet",
        topup.wallet.key,
        "is short",
        topup.shortfall,
        "| bundles:",
        [b.id for b in topup.bundles],
    )
    invoice = topup.top_up()  # the suggested bundle; or top_up("Starter")
    print("pay at:", invoice.checkout_url)
    invoice = invoice.wait_paid(timeout=600)
    print(invoice.status.value, "| balance now:", topup.wallet.refresh().balance)
    plan = plan  # nothing to skip: pre-flight again below passes
else:
    plan = plan.skip(*(row.path for row in pre.held))

pre = plan.preflight()
print("sending:", [r.path for r in pre.sending])
print("skipped:", [r.path for r in pre if r.verdict is syfthub.Verdict.SKIPPED])
print("ready:", pre.ready)

sending: ['federal-office-public-health/federal-office-health-public', 'cantonal-social-services/cantonal-social-services-fsio-public', 'cantonax_tax_office/federal-tax-administration-public', 'cantonal-migration-office/state-secretariat-migration-public']
skipped: []
ready: True


## 5. Execute

One request per Space, in parallel. `results.chunks` ranks scored passages across the document
sources; `results.answers` holds what model sources returned (`results.documents` is both, chunks
first). A metered source is paid on the way: the Space challenges, the Hub signs the credential,
and the charge shows in `results.cost` and `hub.spent`.

In [10]:
results = plan.execute()
for path, doc in results.chunks[:10]:
    title = f" ({doc.title})" if doc.title else ""
    print(f"[{doc.score:.2f}] {path}{title}\n    {doc.content[:200]!r}")
print()
print("answers:", {k: v[:120] for k, v in results.answers.items()})
print("cost:", results.cost, "| spent this session:", hub.spent)


answers: {}
cost: {} | spent this session: {}


In [11]:
for r in results:
    print(
        f"{r.path:45} {r.outcome.value:10} {r.reason.value if r.reason else '':16} "
        f"status={r.status_code} paid_from={r.paid_from or '-'} cost={r.cost or '-'} "
        f"{r.elapsed_ms} ms"
    )
if results.failed:
    print("\nerrors:", {r.path: r.error for r in results.failed})

federal-office-public-health/federal-office-health-public failed     server_error     status=500 paid_from=- cost=- 4078 ms
cantonal-social-services/cantonal-social-services-fsio-public failed     server_error     status=500 paid_from=- cost=- 5071 ms
cantonax_tax_office/federal-tax-administration-public failed     server_error     status=500 paid_from=- cost=- 5311 ms
cantonal-migration-office/state-secretariat-migration-public failed     server_error     status=500 paid_from=- cost=- 4900 ms

errors: {'federal-office-public-health/federal-office-health-public': "Model chat failed: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}", 'cantonal-social-services/cantonal-social-services-fsio-public': "Model chat failed: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}", 'cantonax_tax_office/federal-tax-administration-public': "Model chat failed: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}", 'cantonal-migration-off

## 6. When a Space says no

Rejections and failures never raise at execute time. They land in the row with a `Reason` and the
Space's raw body in `r.raw`, and `results.retryable` names the rows worth sending again:
rate-limited, timed out, server errors, and hosts that were cooling down. `results.retry()`
re-sends only those and merges them in. With nothing retryable it returns the same results.

In [12]:
print("retryable:", [r.path for r in results.retryable])
results = results.retry()
[(r.path, r.outcome.value) for r in results]

retryable: ['federal-office-public-health/federal-office-health-public', 'cantonal-social-services/cantonal-social-services-fsio-public', 'cantonax_tax_office/federal-tax-administration-public', 'cantonal-migration-office/state-secretariat-migration-public']


[('federal-office-public-health/federal-office-health-public', 'rejected'),
 ('cantonal-social-services/cantonal-social-services-fsio-public', 'rejected'),
 ('cantonax_tax_office/federal-tax-administration-public', 'rejected'),
 ('cantonal-migration-office/state-secretariat-migration-public', 'rejected')]

## 7. Wallets

A prepaid source bills a wallet on the publisher's wallet host. `hub.wallets.list()` reads every
wallet behind the sources you can see and its balance, each from its own host, in parallel; a
host that does not answer leaves `balance=None` and is held at pre-flight rather than sent to.

In [13]:
for w in hub.wallets.list():
    print(
        f"{w.key:35} {str(w.balance) if w.balance is not None else 'unavailable':>16}   {', '.join(w.sources)}"
    )

sekretariat/cluster/IDR                        0 IDR   putricintobuliah/news-new, putricintobuliah/newstestingg
shubham2/cluster/PHP                         320 PHP   shubham2/learn-ml
hukumonline/xendit/IDR                   unavailable   hukumonline/hukumonline-news
republika/xendit/IDR                           0 IDR   republika/republika-pusdok-archive, republika/republika-pusdok-dataset
tempo/xendit/IDR                         unavailable   tempo/plus, tempo/news


## 8. Budget

There is no budget until you set one. A budget is one cap per currency, since publishers bill in
their own currencies and the SDK never converts money: `budget=1.00` caps USD only,
`budget={"USD": 1.00, "IDR": 50000}` caps both, and a currency with no cap is not budgeted. Rows
whose estimate would cross the cap in their currency are held with an `OverBudget` action, in plan
order. `execute(strict=True)` raises `ActionRequired` instead of sending partially.

The cell caps every currency the plan would spend at just under its estimate, so each paid source
is held once; with only free sources nothing changes.

In [14]:
estimate = plan.preflight().estimate  # {"CUR": Money} across the sending rows
caps = {cur: money.amount * 0.5 for cur, money in estimate.items()}
hub.set_budget(caps or None)
print("caps:", hub.budget.limits)

pre = plan.preflight()
print([(row.path, row.verdict.value) for row in pre])
for a in pre.actions:
    print(a.kind.value, a.paths, "estimate", getattr(a, "estimate", None))

try:
    plan.execute(strict=True)
except syfthub.ActionRequired as e:
    print("ActionRequired:", e)
hub.set_budget(None)  # no cap again

caps: {}


[('federal-office-public-health/federal-office-health-public', 'ready'), ('cantonal-social-services/cantonal-social-services-fsio-public', 'ready'), ('cantonax_tax_office/federal-tax-administration-public', 'ready'), ('cantonal-migration-office/state-secretariat-migration-public', 'ready')]


Budget(limits={}, spent={})

## 9. Save and resume

Plans, invoices and results are plain data. An app can hand a top-up to the user, save the job, and
pick it up later or in another process. `hub.load` rebinds a snapshot on the sync twin;
`SearchPlan.from_dict(hub, d)` does the same on `AsyncHub`.

In [15]:
import json

saved = json.dumps({"plan": plan.to_dict(), "results": results.to_dict()})
restored = hub.load(syfthub.SearchPlan, json.loads(saved)["plan"])
print(restored == plan, "| the same plan; execute() would send it again")
hub.load(syfthub.Results, json.loads(saved)["results"]).paths

True | the same plan; execute() would send it again


('federal-office-public-health/federal-office-health-public',
 'cantonal-social-services/cantonal-social-services-fsio-public',
 'cantonax_tax_office/federal-tax-administration-public',
 'cantonal-migration-office/state-secretariat-migration-public')

## 10. What the SDK sent

The SDK logs one INFO line per execute and one DEBUG line per request, with secrets redacted, to
the `syfthub` logger. Turn it on to see that queries go to the Spaces and the Hub sees only
catalogue, token and pay calls.

In [16]:
import logging

logging.basicConfig(format="%(name)s %(levelname)s %(message)s")
logging.getLogger("syfthub").setLevel(logging.DEBUG)
_ = plan.execute()
logging.getLogger("syfthub").setLevel(logging.WARNING)

syfthub DEBUG POST https://syfthub.openmined.org/api/v1/endpoints/search -> 200 (attempt 1) headers={'User-Agent': 'syfthub-sdk/2.0.0a1', 'Authorization': '<redacted>'}


syfthub DEBUG POST http://74.161.162.161:8087/api/v1/endpoints/federal-office-health-public/query -> 429 (attempt 1) headers={'User-Agent': 'syfthub-sdk/2.0.0a1', 'Authorization': '<redacted>', 'Content-Type': 'application/json'}


syfthub DEBUG POST http://74.161.162.161:8086/api/v1/endpoints/state-secretariat-migration-public/query -> 429 (attempt 1) headers={'User-Agent': 'syfthub-sdk/2.0.0a1', 'Authorization': '<redacted>', 'Content-Type': 'application/json'}


syfthub DEBUG POST http://74.161.162.161:8085/api/v1/endpoints/federal-tax-administration-public/query -> 429 (attempt 1) headers={'User-Agent': 'syfthub-sdk/2.0.0a1', 'Authorization': '<redacted>', 'Content-Type': 'application/json'}


syfthub DEBUG POST http://74.161.162.161:8088/api/v1/endpoints/cantonal-social-services-fsio-public/query -> 429 (attempt 1) headers={'User-Agent': 'syfthub-sdk/2.0.0a1', 'Authorization': '<redacted>', 'Content-Type': 'application/json'}


syfthub INFO search: 0 returned, 4 failed, 0 held, 0 payment required; cost 0


In [17]:
hub.close()

## Where next

- `examples/openai_chat.py` is the reference app: pre-flight, hand the top-up to the user, resume after payment, compose an answer with OpenAI.
- `docs/guides/python-sdk.md` is the full guide, including the migration table from the 0.x client.
- `syfthub.testing.FakeHub` stands in for a Hub and its Spaces in your own tests, with no network.